# Phase 1 — Data Access + Portfolio Agent

> **Gemini-only project.** Every cell here uses Google Gemini (chat + embeddings) via `langchain-google-genai`. The only key the core needs is `GOOGLE_API_KEY`. This notebook is a **scaffold** — run it top-to-bottom *after* Claude Code finishes this phase. If a cell references a module that doesn't exist yet, that phase hasn't been built.

**Goal:** read the real Excel portfolios and answer holdings + performance questions.

**Data facts used here:** `CLT-002` holds ETFs + individual stocks + cash (ideal for the 3-type distinction). Cash `quantity` is a dollar balance; never multiply it by purchase price.


In [1]:
# --- Setup: load .env and make the project importable ---
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # repo root
from dotenv import load_dotenv
load_dotenv('../.env', override=True)  # override=True: beat VS Code's stale cached env vars

# This project is Gemini-only. Confirm the one required key is present.
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env (see 00_prerequisites_and_setup.md)'
print('GOOGLE_API_KEY loaded:', bool(os.getenv('GOOGLE_API_KEY')))
print('SEC_USER_AGENT   :', os.getenv('SEC_USER_AGENT', '(set before Phase 5)'))

GOOGLE_API_KEY loaded: True
SEC_USER_AGENT   : iampkumar iampkumar03@gmail.com


## 1. Repository reads the real `.xlsx` (sheet 'Potfolios' — sic)

In [2]:
from app.data.repositories import ExcelPortfolioRepository
repo = ExcelPortfolioRepository()
p = repo.get('CLT-002')
print('CLT-002 holdings:', len(p.holdings))
for h in p.holdings:
    print(f'  {h.symbol:6} {h.asset_class:22} qty={h.quantity} is_cash={h.is_cash} is_etf={h.is_etf}')

{"rows": 84, "clients": 10, "event": "portfolios_loaded", "level": "info", "timestamp": "2026-07-12T03:56:21.781845Z"}
CLT-002 holdings: 7
  QQQ    Technology ETF         qty=850.0 is_cash=False is_etf=True
  NVDA   Individual Stock       qty=200.0 is_cash=False is_etf=False
  MSFT   Individual Stock       qty=180.0 is_cash=False is_etf=False
  ARKK   Growth ETF             qty=1200.0 is_cash=False is_etf=True
  VEA    International ETF      qty=1800.0 is_cash=False is_etf=True
  TSLA   Individual Stock       qty=85.0 is_cash=False is_etf=False
  CASH   Cash Equivalent        qty=55429.0 is_cash=True is_etf=False


## 2. Cash is handled correctly (dollar balance, not qty x price)

In [5]:
cash = [h for h in repo.get('CLT-001').holdings if h.is_cash][0]
print('CASH quantity (=$ balance):', cash.quantity)
print('CASH cost_basis          :', cash.cost_basis, '(should equal quantity, NOT quantity*price)')

CASH quantity (=$ balance): 160050.0
CASH cost_basis          : 160050.0 (should equal quantity, NOT quantity*price)


## 2b. Allocation by asset class (stocks vs ETF types vs cash)

In [6]:
from app.tools.portfolio_tools import get_allocation_by_asset_class, get_allocation_by_sector
print(get_allocation_by_asset_class('CLT-002'))
print(get_allocation_by_sector('CLT-002'))

{'client_id': 'CLT-002', 'total_value': 1042358.11, 'allocation_pct': {'Technology ETF': 59.16, 'Individual Stock': 14.02, 'International ETF': 12.26, 'Growth ETF': 9.24, 'Cash Equivalent': 5.32}}
{'client_id': 'CLT-002', 'total_value': 1042358.11, 'allocation_pct': {'Technology': 65.81, 'International': 12.26, 'Innovation': 9.24, 'Cash': 5.32, 'Semiconductors': 4.05, 'Electric Vehicles': 3.33}}


## 3. Portfolio agent answers 'what do I own?' (grouped by asset class)

In [3]:
from app.agents.portfolio import PortfolioAgent
agent = PortfolioAgent()
print(agent.answer(client_id='CLT-002', query='What do I own?'))

{"query": "What do I own?", "event": "agent_start", "client_id": "CLT-002", "agent": "portfolio", "session_id": "adhoc", "level": "info", "timestamp": "2026-07-12T03:56:28.892324Z"}


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


{"rows": 84, "clients": 10, "event": "portfolios_loaded", "client_id": "CLT-002", "agent": "portfolio", "session_id": "adhoc", "level": "info", "timestamp": "2026-07-12T03:56:30.485335Z"}


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


{"seconds": 3.68, "new_messages": 4, "tool_calls": 1, "event": "agent_done", "client_id": "CLT-002", "agent": "portfolio", "session_id": "adhoc", "level": "info", "timestamp": "2026-07-12T03:56:32.573348Z"}
You own the following:

**Individual Stocks:**
* NVDA (NVIDIA Corporation)
* MSFT (Microsoft Corporation)
* TSLA (Tesla Inc)

**ETFs:**
* QQQ (Invesco QQQ Trust ETF) - Technology ETF
* ARKK (ARK Innovation ETF) - Growth ETF
* VEA (Vanguard FTSE Developed Markets ETF) - International ETF

**Cash Equivalents:**
* CASH (High-Yield Savings)


## 4. Performance tools (the brief's YTD + since-purchase questions)

In [4]:
from app.tools.portfolio_tools import get_position_performance, get_ytd_returns
print('NVDA since purchase:', get_position_performance('CLT-002', 'NVDA'))
print('Best YTD return    :', get_ytd_returns('CLT-002'))

NVDA since purchase: {'held': True, 'symbol': 'NVDA', 'purchase_date': '2024-01-16', 'purchase_price': 55.018, 'current_price': 210.96, 'return_pct': 283.44, 'absolute_gain': 31188.4}
Best YTD return    : {'client_id': 'CLT-002', 'as_of': '2026-07-12', 'ytd_returns': [{'symbol': 'QQQ', 'ytd_return_pct': 18.61}, {'symbol': 'VEA', 'ytd_return_pct': 13.11}, {'symbol': 'NVDA', 'ytd_return_pct': 11.84}, {'symbol': 'ARKK', 'ytd_return_pct': 2.47}, {'symbol': 'TSLA', 'ytd_return_pct': -6.92}, {'symbol': 'MSFT', 'ytd_return_pct': -18.21}], 'best': {'symbol': 'QQQ', 'ytd_return_pct': 18.61}, 'worst': {'symbol': 'MSFT', 'ytd_return_pct': -18.21}}


## ✅ Acceptance check

In [ ]:
ans = agent.answer(client_id='CLT-002', query='What do I own?').lower()
assert any(k in ans for k in ['etf', 'fund'])   # names an ETF
assert 'cash' in ans                            # names cash
print('Phase 1 acceptance: PASS')